[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/07_nativeness/07_nativeness_lab.ipynb)


# 07 — Nativeness · Naturalness — AbNatiV · Ab-RoBERTa

> 본문 [`07_nativeness.md`](07_nativeness.md) 와 **한 절씩 짝지어** 보세요.
> **실측 소요 —** Ab-RoBERTa 5후보 × VH+VL **70초** · AbNatiV 스코어링 **3.5~6.6초**(단 체크포인트 내려받기 **약 33분 / 2.6GB** → 기본 비활성)
> **앞 랩에서 이어져요** — Ch.04 · Ch.05 · Ch.06 의 `my_run/` 을 먼저 찾고, 없으면 커밋된 `data/` 로 대신합니다.

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `07_nativeness/` 로 이동해 필요한 패키지만 깝니다(로컬에서 `07_nativeness/` 안에 열었다면 클론 생략).
끝나면 **`my_run/`**(내가 만들 결과)과 **`data/`**(커밋된 레퍼런스)가 준비돼요 — 랩은 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며 어느 쪽을 썼는지 매번 print 합니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "07_nativeness"
APT_PKGS = ""     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib torch transformers"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True, timeout=None, quiet=False):
    # timeout 을 꼭 주세요 — 네트워크가 '멈춘 채' 매달리면 예외가 안 나서 data/ 폴백이 안 걸립니다.
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        return subprocess.run(cmd, shell=True, check=check, timeout=timeout)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        if check:
            raise subprocess.CalledProcessError(p.returncode, cmd)
    return p

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

_APT = APT_PKGS.split() if (APT_PKGS and IN_COLAB) else []   # 모아서 apt 를 한 번만 돌립니다
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    _APT.append("fonts-nanum")             # Colab 엔 한글 폰트가 없어 라벨이 □ 로 깨져요

if _APT:                                   # apt 인덱스 갱신은 한 번만 (ANARCI 는 hmmscan 이 필요해요)
    _run("apt-get -qq update", timeout=600, quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y " + " ".join(_APT), timeout=900, quiet=True)


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — Ab-RoBERTa pseudo-likelihood (본문 7.2.1)

항체 전용 언어모델 `mogam-ai/Ab-RoBERTa` 로 각 position 을 차례로 `<mask>` 로 가리고, **그 자리에 있던 진짜 잔기의 로그확률**을 모아 평균해요(masked pseudo-log-likelihood). 결과 CSV 는 `variant · chain · mean_logp · perplexity · n_residues` 다섯 컬럼이에요.

- `mean_logp` — 마스킹한 자리의 평균 로그확률. **높을수록** 자연스러움
- `perplexity = exp(-mean_logp)` — 같은 값을 뒤집어 본 것. **낮을수록** 좋음
- `chain` — `VH` · `VL` 각각과, 둘을 **길이가중 평균**한 `paired`. 비교는 `paired` 로 해요

스코어링 스크립트는 이 챕터 폴더에 함께 실려 있어요 — `score_abroberta_pseudolikelihood.py`.

```bash
python score_abroberta_pseudolikelihood.py --fasta variants.fasta --out scores.csv
```

후보는 **앞 랩에서 내가 만든 것**을 먼저 씁니다(Ch.05 Sapiens · Ch.06 Humatch/AnthroAb). 빠진 후보만 `data/variants.fasta` 로 채워요 —
이 노트북의 모든 표는 `my_run/` 을 먼저 찾고 없으면 `data/` 로 폴백하며, 어느 쪽을 썼는지 그때그때 print 합니다.

In [ ]:
import pandas as pd

PREV = [
    ("05_humanize_sapiens", "sapiens_humanized_noguard.fasta", ("sapiens_humanized_VH", "sapiens_humanized_VL"), "sapiens"),
    ("06_cdr_safe_tools",   "humatch_humanised.fasta",         ("humatch_humanised_VH", "humatch_humanised_VL"), "humatch"),
    ("06_cdr_safe_tools",   "anthroab_best_score.fasta",       ("anthroab_bestscore_VH", "anthroab_bestscore_VL"), "anthroab"),
    ("06_cdr_safe_tools",   "anthroab_masked_FRonly.fasta",    ("anthroab_masked_VH", "anthroab_masked_VL"), "anthroabFRmasked"),
]

cands, SRC = {"parental": (VH, VL)}, {"parental": "data/parental.fasta"}
for chapter, fname, keys, label in PREV:
    p = GUIDE_ROOT / chapter / "my_run" / fname
    if p.exists():
        f = read_fasta(p)
        if keys[0] in f and keys[1] in f:
            cands[label], SRC[label] = (f[keys[0]], f[keys[1]]), f"내 결과 · {chapter}/my_run/{fname}"

need = [lab for *_, lab in PREV if lab not in cands]      # 빠진 후보만 레퍼런스로 채워요
if need:
    vp = find_one("variants.fasta", quiet=True)
    v = read_fasta(vp)
    for lab in need:
        cands[lab], SRC[lab] = (v[f"{lab}_VH"], v[f"{lab}_VL"]), f"레퍼런스 · {vp}"

for k in cands:
    print(f"  {k:18s} VH {len(cands[k][0]):3d} aa · VL {len(cands[k][1]):3d} aa  ← {SRC[k]}")

fa = write_fasta(MY / "variants.fasta",
                 {f"{n}_{ch}": s for n, (h, l) in cands.items() for ch, s in (("VH", h), ("VL", l))})
print(f"\n후보 {len(cands)}종 →", fa)

In [ ]:
t0, rows, abr = time.time(), [], None
try:
    # import 도 try 안에 둬요 — torch/transformers 가 어긋나면 여기서 끊고 레퍼런스로 이어가야
    # 뒤 셀들이 연쇄로 죽지 않아요.
    from score_abroberta_pseudolikelihood import score_paired
    for name, (h, l) in cands.items():
        res = score_paired(h, l)
        for ch in ("VH", "VL", "paired"):
            rows.append({"variant": name, "chain": ch,
                         "mean_logp": res[ch]["mean_logp"],
                         "perplexity": res[ch]["perplexity"],
                         "n_residues": res[ch]["n"]})
        print(f"{name:18s} paired mean_logp={res['paired']['mean_logp']:+.4f}  "
              f"ppl={res['paired']['perplexity']:.4f}")
    abr = pd.DataFrame(rows)
    abr.to_csv(MY / "abroberta_scores_summary.csv", index=False)
    print(f"\nAb-RoBERTa 스코어링 {time.time()-t0:.1f}초 (실측 70초 · 5후보 × VH+VL) →",
          MY / "abroberta_scores_summary.csv")
except Exception as e:
    print("Ab-RoBERTa 실행 실패:", type(e).__name__, str(e)[:200])
    print("모델 다운로드가 막힌 거라면 셀을 다시 실행하면 이어받아요. 지금은 레퍼런스로 진행해요.")

if abr is None:
    abr = pd.read_csv(find_one("abroberta_scores_summary.csv"))

rank = abr[abr.chain == "paired"].sort_values("mean_logp", ascending=False)
print("\npaired 순위(자연스러운 순) —",
      " > ".join(f"{r.variant} {r.mean_logp:+.4f}" for r in rank.itertuples()))

## 2) 내 결과 확인 — perplexity 로 자르기 (본문 7.2.2)

`paired` 는 VH·VL 을 길이로 가중한 값이라 체인 하나가 무너져도 절반쯤 희석돼요. 그래서 **체인별 값을 먼저 펼쳐 놓고** paired 를 봐요.

판정은 절대값이 아니라 **parental 대비 배수**로 합니다. 항체마다 서열 자체의 난이도가 달라서, 같은 파이프라인 안의 parental 이 가장 정직한 기준선이거든요.

In [ ]:
def keep_cols(df, wanted):
    """있는 컬럼만 골라내요 — 하드코딩한 목록을 그대로 넘기면 KeyError 로 죽어요."""
    return [x for x in wanted if x in df.columns]

pv  = abr.pivot_table(index="variant", columns="chain", values="mean_logp")
ppl = abr[abr.chain == "paired"].set_index("variant")["perplexity"]
order = [n for n in cands if n in pv.index] or list(pv.index)

tbl = pv.reindex(order)[keep_cols(pv, ["VH", "VL", "paired"])].round(4)
tbl["perplexity(paired)"] = ppl.reindex(order).round(4)
display(tbl)

vh = pv["VH"].dropna().sort_values(ascending=False)
print(f"VH 만 보면 가장 자연스러운 후보는 {vh.index[0]} ({vh.iloc[0]:+.4f}) 이에요.")
if vh.index[0] == "parental":
    print("사람다움을 올린 편집이 VH 의 자연스러움은 깎았어요 — humanness 와 naturalness 는 다른 축이에요.")

base = float(ppl["parental"]) if "parental" in ppl.index else float(ppl.median())
print(f"\n기준선 parental paired perplexity {base:.4f} · 이번 실행의 paired 범위 "
      f"{ppl.min():.4f}~{ppl.max():.4f}")
print("컷오프 — 1.0배 이하 통과 · 1.0~2.0배 주의 · 2.0배 이상이면 서열이 무너진 신호로 봐요.")
warn = []
for n in order:
    r = float(ppl[n]) / base
    v_ = "통과" if r <= 1.0 else ("주의" if r < 2.0 else "★경고")
    if r >= 2.0:
        warn.append(n)
    print(f"  {n:18s} ppl {float(ppl[n]):.4f} · parental 대비 {r:.2f}배 → {v_}")
print("\n판정 —", ("2.0배를 넘긴 후보 " + ", ".join(warn) + " 는 naturalness 축에서 먼저 걸러요."
                  if warn else "2.0배를 넘긴 후보가 없어 naturalness 축에서 즉시 탈락하는 후보는 없어요."))

In [ ]:
from humanization_viz import humanness_bars
from IPython.display import Image, display
import numpy as np

# exp(mean_logp) = 잔기당 pseudo-likelihood 기하평균(0~1) → 막대로 보기 좋은 형태
others = [v for v in pv.index if v != "parental"]
pick = "sapiens" if "sapiens" in others else others[0]
CH = keep_cols(pv, ["VH", "VL", "paired"])
bars = [{"chain": ch, "parental": float(np.exp(pv.loc["parental", ch])),
         "humanized": float(np.exp(pv.loc[pick, ch]))} for ch in CH]
png = humanness_bars(bars, f"Ab-RoBERTa naturalness — parental vs {pick} (exp(mean logP))",
                     MY / "07_naturalness.png", ylabel="per-residue pseudo-likelihood")
display(Image(str(png)))

up   = [ch for ch in CH if pv.loc[pick, ch] > pv.loc["parental", ch]]
down = [ch for ch in CH if pv.loc[pick, ch] <= pv.loc["parental", ch]]
print(f"판정 — {pick} 는 {', '.join(up) or '없음'} 에서 parental 보다 자연스러워졌고, "
      f"{', '.join(down) or '없음'} 에서는 내려갔어요.")
print("체인 하나만 내려가도 paired 는 올라갈 수 있어요. naturalness 는 paired 만 보지 말고 체인별로 확인하세요.")

## 3) 직접 실행 — AbNatiV nativeness (본문 7.1.1~7.1.2) · **기본 비활성**

AbNatiV 는 "사람 잔기를 얼마나 썼나"(humanness)가 아니라 **"그 조합이 실제 사람 항체로서 얼마나 자연스러운가"**(nativeness)를 봐요.

설치는 한 줄인데 **두 번 막혀요.**

```bash
conda create -n abnativ -c conda-forge python=3.10 -y && conda activate abnativ
python -m pip install abnativ            # abnativ 2.0.8 (ImmuneBuilder 동반)
abnativ init                             # 함정① 체크포인트(모델당 약 1GB) — 안 받으면 score 가 FileNotFoundError: .../pretrained_models/vh_model.ckpt
conda install -c bioconda -c conda-forge anarci hmmer -y   # 함정② -align 이 anarci 를 import (Humatch 와 같은 에러)
```

옵션 두 개도 짚고 가요 — **`-align`** 은 ANARCI 로 번호를 매겨 FR·CDR 구간을 나눠 주고(그래서 anarci·hmmscan 이 필요), **`-mean`** 은 잔기별 점수를 서열당 한 값으로 접어 줘요. 이 분해 덕분에 overall 뿐 아니라 FR·CDR 을 따로 볼 수 있어요.

**스코어링 자체는 4서열에 3.5~6.6초로 끝나요.** 무거운 건 체크포인트예요 — `abnativ init` 은 9개 ckpt(~6GB)를 **순차로** 받아 매우 느리고(실측 30분에 3개), 실제로 쓰는 4개만 **병렬로** 받아도 **약 33분 / 2.6GB**(실측)예요.

그래서 아래 셀은 **`RUN_ABNATIV = False` 가 기본**입니다. `True` 로 바꾸면 받은 결과를 `my_run/abnativ_summary_all_models.csv` 로 합쳐 두고, 다음 절이 그걸 먼저 집어요.

In [ ]:
import re

RUN_ABNATIV = False        # ← True 로 바꾸면 체크포인트를 받아 실제로 스코어링해요

# (nat 이름, 입력 fasta, oid, ckpt 파일명)
ABN_MODELS = [("VH", "abnativ_vh.fa", "vh", "vh_model"), ("VLambda", "abnativ_vl.fa", "vl", "vlambda_model"),
              ("VH2", "abnativ_vh.fa", "vh2", "vh2_model"), ("VL2", "abnativ_vl.fa", "vl2", "vl2_model")]

def abnativ_merge(outdir):
    """abnativ 가 남긴 *_seq_scores.csv 들을 summary 한 장으로. 모델·세대는 파일명이 아니라
    컬럼 이름('AbNatiV VH2 Score')에서 읽어요 — oid 를 어떻게 주든 안 깨지게."""
    out = []
    for p in sorted(pathlib.Path(outdir).glob("*_seq_scores.csv")):
        d = pd.read_csv(p)
        hit = [re.match(r"AbNatiV (\w+) Score$", x) for x in d.columns]
        m = next((h.group(1) for h in hit if h), None)
        if m is None:
            continue
        for rec in d.to_dict("records"):
            out.append({"model_generation": "AbNatiV2" if m.endswith("2") else "AbNatiV1",
                        "abnativ_model": m, "variant": rec.get("seq_id"),
                        "overall_score": rec.get(f"AbNatiV {m} Score"),
                        "FR_score":   rec.get(f"AbNatiV FR-{m} Score"),
                        "CDR1_score": rec.get(f"AbNatiV CDR1-{m} Score"),
                        "CDR2_score": rec.get(f"AbNatiV CDR2-{m} Score"),
                        "CDR3_score": rec.get(f"AbNatiV CDR3-{m} Score")})
    return pd.DataFrame(out)

if RUN_ABNATIV:
    try:
        _ensure("abnativ anarci")
        if IN_COLAB and not shutil.which("hmmscan"):
            _run("apt-get -qq install -y hmmer", check=False)      # -align → anarci → hmmscan
        home = pathlib.Path.home() / ".abnativ/models/pretrained_models"
        home.mkdir(parents=True, exist_ok=True)
        base_url = "https://zenodo.org/record/17295347/files"
        need = [ck for *_, ck in ABN_MODELS if not (home / f"{ck}.ckpt").exists()]
        if need:   # 순차 abnativ init 대신 필요한 것만 병렬로. 멈춤은 예외가 안 나니 타임아웃·재시도 필수
            par = " ".join(f'wget -c -q --timeout=30 --tries=3 -O "{home}/{ck}.ckpt" '
                           f'"{base_url}/{ck}.ckpt?download=1" &' for ck in need)
            _run(f"({par} wait)", check=False)
            miss = [ck for ck in need if not (home / f"{ck}.ckpt").exists()]
            assert not miss, f"체크포인트를 못 받았어요 {miss} — 셀을 다시 실행하면 -c 로 이어받아요."

        write_fasta(MY / "abnativ_vh.fa", {f"{n}_VH": h for n, (h, l) in cands.items()})
        write_fasta(MY / "abnativ_vl.fa", {f"{n}_VL": l for n, (h, l) in cands.items()})
        t0 = time.time()
        for nat, fa_in, oid, _ck in ABN_MODELS:
            _run(f'CUDA_VISIBLE_DEVICES="" abnativ score -nat {nat} -i "{MY/fa_in}" '
                 f'-odir "{MY/"abnativ"}" -oid {oid} -align -ncpu 4')
        print(f"AbNatiV 스코어링 {time.time()-t0:.1f}초 (실측 3.5~6.6초/모델)")

        merged = abnativ_merge(MY / "abnativ")
        if len(merged):
            merged.to_csv(MY / "abnativ_summary_all_models.csv", index=False)
            print(f"세대 {merged.model_generation.nunique()}종 · {len(merged)}행 합쳤어요 →",
                  MY / "abnativ_summary_all_models.csv")
        else:
            print("*_seq_scores.csv 를 못 찾았어요 — 다음 절은 커밋된 레퍼런스로 이어가요.")
    except Exception as e:
        print("AbNatiV 실행 실패:", type(e).__name__, str(e)[:200])
        print("다음 절은 커밋된 레퍼런스로 이어가요.")
else:
    print("RUN_ABNATIV = False → 커밋된 레퍼런스로 진행해요.")
    print("스코어링은 3.5~6.6초로 빠르지만 체크포인트가 약 33분 / 2.6GB 예요(실측).")

## 4) 내 결과 / 레퍼런스 — 두 세대를 나란히 (본문 7.1.3 · 7.2.3)

AbNatiV 에는 **세대가 둘**이고, 본문의 두 표는 서로 다른 모델이에요. 세대를 섞으면 같은 후보인데 숫자가 모순돼 보여요.

| 세대 | 옵션 | 본문 |
|---|---|---|
| AbNatiV1 | `-nat VH` / `-nat VLambda` | 7.1.3 표 |
| AbNatiV2 | `-nat VH2` / `-nat VL2` | 7.2.3 표의 AbNatiV2 열 |

두 세대를 **같은 후보 집합으로 나란히** 그려서, 무엇이 세대 차이이고 무엇이 후보 차이인지 갈라 봐요.

In [ ]:
abn = pd.read_csv(find_one("abnativ_summary_all_models.csv"))

SCORES = ["overall_score", "FR_score", "CDR1_score", "CDR2_score", "CDR3_score"]

def cand_of(v):
    """'sapiens_humanized_VH' · 'sapiens_VH' 어느 표기든 후보 키로."""
    s = re.sub(r"_(VH|VL)$", "", str(v))
    return s if s in cands else s.split("_")[0]

abn["candidate"] = abn.variant.map(cand_of)
abn["chain"] = abn.variant.map(lambda v: "VH" if str(v).endswith("_VH") else "VL")

for gen, sub in abn.groupby("model_generation"):
    print(f"=== {gen} ===")
    display(sub[keep_cols(sub, ["abnativ_model", "variant"] + SCORES)].round(4).reset_index(drop=True))

def val(gen, chain, cand, col):
    s = abn[(abn.model_generation == gen) & (abn.chain == chain) & (abn.candidate == cand)]
    return float(s.iloc[0][col]) if len(s) else float("nan")

ref_cand = "sapiens" if "sapiens" in set(abn.candidate) else sorted(set(abn.candidate) - {"parental"})[0]
print(f"\nAbNatiV1 VH  parental → {ref_cand}")
deltas = {}
for col, lab in (("overall_score", "overall"), ("FR_score", "FR    "), ("CDR3_score", "CDR-H3")):
    a, b = val("AbNatiV1", "VH", "parental", col), val("AbNatiV1", "VH", ref_cand, col)
    deltas[lab.strip()] = b - a
    print(f"  {lab}  {a:.4f} → {b:.4f}  (Δ {b-a:+.4f})")
vl_par, vl_ref = val("AbNatiV1", "VL", "parental", "overall_score"), val("AbNatiV1", "VL", ref_cand, "overall_score")
print(f"AbNatiV1 VL(lambda) parental {vl_par:.4f} → {ref_cand} {vl_ref:.4f}")
print(f"판정 — overall 상승분 {deltas['overall']:+.4f} 중 FR 이 {deltas['FR']:+.4f}, CDR-H3 가 {deltas['CDR-H3']:+.4f} 예요. "
      + ("상승이 FR 에 몰린 'framework 만 사람화' 프로파일이에요." if deltas["FR"] > deltas["CDR-H3"]
         else "상승이 FR 밖에서도 나왔어요 — CDR 쪽 변화를 서열로 확인하세요."))
print(f"VL 은 parental 이 이미 {vl_par:.4f} 라 올릴 여지가 {1-vl_par:.4f} 밖에 없어요.")

neg = abn[(abn[keep_cols(abn, SCORES)] < 0).any(axis=1)]
if len(neg):
    print(f"\nAbNatiV2 는 CDR 점수가 음수로 내려가요 — 음수가 든 행 {len(neg)}개 · 최소값 "
          f"{abn[keep_cols(abn, SCORES)].min().min():.4f}")
    display(neg[keep_cols(neg, ["model_generation", "variant", "CDR1_score", "CDR2_score", "CDR3_score"])].round(4))
    print("짧은 CDR 은 정렬 구간이 좁아 세대마다 스케일이 크게 달라져요. **세대 간 비교는 overall 열로만** 하세요.")

In [ ]:
# CDR-H3 에 실제로 변이가 들어갔는지는 점수가 아니라 서열로 확인해요
ct = pd.read_csv(find_prev("04_sequence_qc", "cdr_table_imgt.csv",
                           ref=GUIDE_ROOT / "04_sequence_qc" / "data", quiet=True))
h3 = ct[(ct.chain == "H") & (ct.cdr == "CDR3")].iloc[0]["sequence"]
st = VH.find(h3) + 1
assert st > 0, "CDR-H3 를 parental VH 에서 못 찾았어요 — cdr_table 과 parental.fasta 가 어긋나요."
en = st + len(h3) - 1

SEQ_COL = f"CDR-H3 (raw {st}~{en})"
par_h3 = val("AbNatiV1", "VH", "parental", "CDR3_score")
recs = [{"후보": "parental", SEQ_COL: h3, "변이": "-", "n_mut": 0, "AbNatiV1 CDR-H3": par_h3}]
for name, (h, l) in cands.items():
    if name == "parental" or len(h) != len(VH):
        continue
    muts = [f"{a}{i+1}{b}" for i, (a, b) in enumerate(zip(VH, h)) if a != b and st <= i + 1 <= en]
    recs.append({"후보": name, SEQ_COL: h[st-1:en], "변이": ", ".join(muts) or "-", "n_mut": len(muts),
                 "AbNatiV1 CDR-H3": val("AbNatiV1", "VH", name, "CDR3_score")})
h3tbl = pd.DataFrame(recs)
display(h3tbl.round(4))

moved = h3tbl[(h3tbl["n_mut"] > 0) & h3tbl["AbNatiV1 CDR-H3"].notna()]
for _, r in moved.iterrows():
    print(f"{r['후보']:18s} CDR-H3 변이 {int(r['n_mut'])}개 ({r['변이']}) · "
          f"점수 {par_h3:.4f} → {r['AbNatiV1 CDR-H3']:.4f} (Δ {r['AbNatiV1 CDR-H3']-par_h3:+.4f})")
if len(moved):
    big = float((moved["AbNatiV1 CDR-H3"] - par_h3).abs().max())
    print(f"\n판정 — CDR-H3 에 변이가 들어간 후보가 {len(moved)}개인데 점수 변화는 최대 {big:.4f} 예요.")
else:
    print("\n판정 — 이 후보 집합에서는 CDR-H3 에 들어간 변이가 없어요.")
print("CDR 점수가 평평하다고 해서 CDR 을 안 건드렸다는 뜻이 아니에요. 변이 여부는 반드시 서열로 확인하세요.")
print("구조가 실제로 얼마나 움직였는지는 Ch.08 의 CDR-H3 RMSD 로 따로 봐요.")

In [ ]:
from humanization_viz import nativeness_panel
from IPython.display import Image, display

pngs, ranks = [], {}
for gen in [g for g in ["AbNatiV1", "AbNatiV2"] if g in set(abn.model_generation)]:
    sub = abn[(abn.model_generation == gen) & (abn.chain == "VH")]
    rows = [{"label": r.candidate, "overall": r.overall_score, "fr": r.FR_score,
             # 음수 CDR 은 그리지 않아요 — 패널 y축이 0~1.05 라 잘려서 0 처럼 보여요
             "cdr": r.CDR3_score if r.CDR3_score == r.CDR3_score and r.CDR3_score >= 0 else None}
            for r in sub.itertuples()]
    p = nativeness_panel(rows, f"{gen} VH nativeness — overall / FR / CDR-H3",
                         MY / f"07_nativeness_{gen.lower()}.png")
    pngs.append(p)
    ranks[gen] = list(sub.sort_values("overall_score", ascending=False).candidate)

for p in pngs:
    display(Image(str(p)))
for gen, o in ranks.items():
    print(f"{gen} VH overall 순위 —", " > ".join(o))
if len(ranks) == 2:
    a, b = ranks["AbNatiV1"], ranks["AbNatiV2"]
    print("판정 — 두 세대의 1위는 " + ("같지만" if a[0] == b[0] else "다르고") +
          (" 나머지 순위가 뒤집혀요." if a != b else " 순위도 같아요."))
    print("세대가 다르면 값의 스케일도 순위도 달라져요. AbNatiV 점수는 **세대 표기와 함께** 인용하세요.")

## 5) 세 축을 한 표에 — humanness · nativeness · naturalness (본문 7.2.3)

셋은 서로 다른 것을 재요.

| 축 | 지표 | 무엇을 보나 |
|---|---|---|
| humanness | Humatch CNN (H/L) | 사람 germline 계열로 분류되나 |
| nativeness | AbNatiV2 (VH2/VL2) | 자연 사람 레퍼토리에 들어맞나 |
| naturalness | Ab-RoBERTa paired | 언어모델이 이 서열을 자연스럽다고 보나 |

Humatch CNN 은 **자기 산출물에만 붙는 자체 점수**라 다른 후보 칸은 비어요(본문 7.2.3 표와 같아요). OASis 백분위는 수십 GB OAS DB 가 필요해 이 환경에서 재현할 수 없어 뺐고, 그 자리는 Ch.05 Sapiens 확률로 봐요.

In [ ]:
# humanness — 내 Humatch 실행 결과가 있으면 그것, 없으면 커밋된 config 표
hm_out = GUIDE_ROOT / "06_cdr_safe_tools" / "my_run" / "humatch_out.csv"
if hm_out.exists():
    r0 = pd.read_csv(hm_out).iloc[0]
    cnn_h, cnn_l = float(r0["CNN_H"]), float(r0["CNN_L"])
    print(f"[내 결과 · 06_cdr_safe_tools] {hm_out}")
else:
    cfg = pd.read_csv(find_prev("06_cdr_safe_tools", "humatch_scores.csv",
                                ref=GUIDE_ROOT / "06_cdr_safe_tools" / "data")).set_index("metric")["value"]
    cnn_h, cnn_l = float(cfg["CNN_H"]), float(cfg["CNN_L"])

# CNN 이 안 붙는 후보의 humanness 는 Ch.05 Sapiens 확률로 봐요(본문 7.2.3 이 OASis 를 뺀 자리)
try:
    hs = pd.read_csv(find_prev("05_humanize_sapiens", "humanness_summary.csv",
                               ref=GUIDE_ROOT / "05_humanize_sapiens" / "data", quiet=True))
    hs = hs.set_index(["chain", "definition"])["mean_probability"]
    for chn in ("VH", "VL"):
        print(f"Sapiens humanness (Ch.05) {chn} — parental {hs[(chn, 'parental')]:.4f} → "
              f"humanized {hs[(chn, 'definition_b_rescored_humanized')]:.4f}")
except Exception as e:
    print("Ch.05 humanness 표를 못 읽었어요:", type(e).__name__, str(e)[:120])

abn2 = abn[abn.model_generation == "AbNatiV2"].pivot_table(index="candidate", columns="chain",
                                                           values="overall_score")
def gen_col(df, name):
    """세대나 체인이 통째로 빠져도 KeyError 로 죽지 않게."""
    return df[name] if name in getattr(df, "columns", []) else pd.Series(dtype=float)

abr_p = abr[abr.chain == "paired"].copy()
abr_p["candidate"] = abr_p.variant.map(cand_of)
abr_p = abr_p.set_index("candidate")["mean_logp"]

idx = [n for n in cands if n in abr_p.index]
panel = pd.DataFrame({
    "Humatch CNN H/L (humanness↑)": ["%.3f / %.3f" % (cnn_h, cnn_l) if n == "humatch" else "—" for n in idx],
    "AbNatiV2 VH (nativeness↑)": [gen_col(abn2, "VH").get(n, float("nan")) for n in idx],
    "AbNatiV2 VL (nativeness↑)": [gen_col(abn2, "VL").get(n, float("nan")) for n in idx],
    "Ab-RoBERTa paired (naturalness↑)": [abr_p.get(n, float("nan")) for n in idx],
}, index=idx).round(4)

nat_cols = keep_cols(panel, ["AbNatiV2 VH (nativeness↑)", "AbNatiV2 VL (nativeness↑)"])
drop = [n for n in panel.index if panel.loc[n, nat_cols].isna().any()]
if drop:
    print("AbNatiV 점수가 없는 후보는 표에서 뺐어요 —", ", ".join(drop),
          "(AbNatiV 를 이 후보까지 돌리면 채워져요)")
panel = panel.drop(index=drop)
display(panel)

if "parental" in panel.index and len(panel) > 1:
    nat, natu = panel[nat_cols[0]], panel["Ab-RoBERTa paired (naturalness↑)"]
    b_nat, b_natu = float(nat["parental"]), float(natu["parental"])
    outliers = [n for n in panel.index
                if n != "parental" and float(nat[n]) > b_nat and float(natu[n]) < b_natu]
    print(f"\n이상치 규칙 — parental 대비 nativeness 는 올랐는데(> {b_nat:.4f}) "
          f"naturalness 는 내려간(< {b_natu:.4f}) 후보")
    for n in outliers:
        print(f"  ★ {n}  nativeness {float(nat[n])-b_nat:+.4f} · naturalness {float(natu[n])-b_natu:+.4f}")
    print("판정 —", (", ".join(outliers) + " 가 이상치예요. 주 패널(Humatch CNN + AbNatiV2)은 통과시키지만 "
                    "Ab-RoBERTa 가 붙잡는 후보라, 실험 전에 한 번 더 들여다볼 자리예요."
                    if outliers else "주 패널과 Ab-RoBERTa 가 같은 방향을 가리켜 걸러낼 이상치가 없어요."))
else:
    print("\n비교할 후보가 부족해요 — AbNatiV 를 후보 전체에 돌리면 이상치 판정이 켜져요.")
print("역할 분담 — 주 human-likeness 패널은 Humatch CNN + AbNatiV2, Ab-RoBERTa 는 이상치 탐지 보조예요.")

## 이 랩에서 확인한 것

1. **AbNatiV 는 세대를 구분해 적어요.** AbNatiV1 VH **0.6477 → 0.8803**(FR **0.6317 → 0.9245**), VL(lambda) parental **0.9022**. 같은 후보를 AbNatiV2 로 재면 VH **0.4927 → 0.7777** 로 스케일이 통째로 달라요.
2. **CDR-H3 점수가 평평해도 CDR 이 그대로인 건 아니에요.** Sapiens 후보는 CDR-H3 에 **L104D · Y109V 2개**를 넣었는데도 AbNatiV1 CDR-H3 는 **0.6137 → 0.6265**(Δ +0.0127) 로 거의 안 움직였어요. AnthroAb 도 **Y102S · Y109V 2개**에 Δ +0.0834 뿐이에요. 변이 여부는 **서열로** 확인하세요.
3. **AbNatiV2 의 CDR 점수는 음수가 나와요**(실측 최소 **−0.2564**). 구간이 짧을수록 세대별 스케일 차가 커지니 **세대 간 비교는 overall 열로만** 하세요.
4. **naturalness ≠ humanness** — VH 만 보면 parental(**−0.5188**)이 가장 자연스러워요. paired perplexity 는 **1.6444~4.1467** 범위이고 parental **2.0627** 이 기준선이에요. **2.0배**(FR-masked 후보)를 넘기면 서열이 무너진 신호예요.
5. **세 축을 한 표에** 놓으면 어긋나는 후보가 보여요 — humanness(Humatch CNN **0.972 / 1.000**)·nativeness(AbNatiV2)는 올랐는데 naturalness 만 내려간 후보가 Ab-RoBERTa 가 잡아내는 이상치예요.
6. 산출물 — `my_run/abroberta_scores_summary.csv` · `abnativ_summary_all_models.csv`(직접 실행 시) · `07_naturalness.png` · `07_nativeness_abnativ1.png` · `07_nativeness_abnativ2.png`.

다음 → **[Ch.08 — 구조 검증](../08_structure/08_structure_lab.ipynb)**